# Structured output with Pydantic

Define the fields a bank help desk needs in a Pydantic model, then pass that model to `with_structured_output`. LangChain returns a validated `BankRequest` instance instead of unstructured text.

This example uses `gpt-5.4-mini`. Set `OPENAI_API_KEY` in your environment or repository `.env` file before running it.

In [4]:
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from pydantic import BaseModel, Field
from typing import Literal

load_dotenv()


class BankRequest(BaseModel):
    """Classified customer request for bank help desk."""

    intent: Literal[
        "CARD_UNBLOCK",
        "CARD_BLOCK",
        "BALANCE_ENQUIRY",
        "LOAN_QUERY",
        "OTHER"
    ] = Field(description="Customer's main request")

    department: Literal[
        "CARDS", "ACCOUNTS", "LOANS", "GENERAL"
    ] = Field(description="Correct bank counter")

    account_last4: str | None = Field(
        default=None,
        description="Only the last four digits if supplied"
    )

    priority: Literal["LOW", "MEDIUM", "HIGH"]
    needs_human: bool


In [5]:
model = ChatOpenAI(model="gpt-5.4-mini", temperature=0)

model_with_structure = model.with_structured_output(BankRequest)

In [ ]:
customer_message = (
    "My debit card is blocked and I need it unblocked. "
    "The last four digits are 1234."
)
response = model_with_structure.invoke(
    f"Classify this customer request using the BankRequest schema: {customer_message}"
)
print(response.model_dump())

print(f"Intent: {response.intent}")
print(f"Department: {response.department}")
print(f"Account Last 4: {response.account_last4}")
print(f"Priority: {response.priority}")
print(f"Needs Human: {response.needs_human}")


if response.needs_human:
    print("This request needs to be escalated to a human agent.")

{'intent': 'CARD_UNBLOCK', 'department': 'CARDS', 'account_last4': '1234', 'priority': 'HIGH', 'needs_human': True}
Intent: CARD_UNBLOCK
Department: CARDS
Account Last 4: 1234
Priority: HIGH
Needs Human: True
